# Gauge usage and Boundless DAS (Appendix H)

**Paper section.** Appendix H (Gauge usage and comparison to Boundless DAS) with Figure 2, referenced from Section 3 (Boundless DAS and gauge freedoms).

**Claim.** In the paper's words, "Freely trained models use both freedoms mildly. Across models, the mean singular-value ratio is 1.25, and the mean log-RMS variance is 0.08", "At resid_final, Boundless DAS closely recovers the derived causal subspace", and at resid_post "the additional directions that Boundless DAS found affect the output through the orthogonal complement's length, not through the directions themselves".

**Experiment.** The singular-value ratio κ(G) and the variance of log rms(h_post) of every model, the coordinates of one free and one pinned model in their derived causal subspaces, Boundless DAS at resid_final and resid_post, and the rescaling of the orthogonal component of resid_post to three times its length, from `scripts/run_gauge_boundless_das.sh`.

**Saved Outputs.** In `figures/`: `figure2.pdf`. The notebook also prints the numbers of Appendix H.

**How the notebook works.** The first code cell sets the paths and defines the loaders. The gauge-usage cell prints the means over the free and over the pinned models. The Figure 2 cell computes the ground-truth log next-token probabilities of the nonergodic_c2a06 process on the evaluation contexts by exact filtering, applies the example gauges of each hook, and draws them next to the coordinates of the free and the pinned model with four layers, four heads, and seed 1. The last two cells print the Boundless DAS medians and the effect of the rescaling.

In [ ]:
import glob, os, re, sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# ═══════════════════════════════════════════════════════════
# CONFIG: change these paths, everything else follows
# ═══════════════════════════════════════════════════════════
RESULTS_DIR = '../results'
PLOT_DIR = '../figures'
os.makedirs(PLOT_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════
# Style
# ═══════════════════════════════════════════════════════════
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'font.weight': 'light',
})
sns.set_context('notebook')

# ═══════════════════════════════════════════════════════════
# Loaders
# ═══════════════════════════════════════════════════════════
def load(name, variant):
    """Every group of one experiment and one variant ('free' or 'pinned'), concatenated over models."""
    out = {}
    for f in sorted(glob.glob(os.path.join(RESULTS_DIR, name, variant, f'{name}_v*.npz'))):
        z = np.load(f, allow_pickle=True)
        for k in z.files:
            if z[k].ndim >= 1:
                out.setdefault(k, []).append(z[k])
    return {k: np.concatenate(v) for k, v in out.items()}

def load_all(name):
    """The free and the pinned models together."""
    a, b = load(name, 'free'), load(name, 'pinned')
    return {k: np.concatenate([a[k], b[k]]) for k in a if k in b}

def process_of(tag):
    """The process name of a model tag such as mess3_a85x05_4l_h4_s0."""
    return re.sub(r'_\d+l_h\d+_s\d+$', '', str(tag))

## Appendix H: gauge usage

In [ ]:
for variant in ['free', 'pinned']:
    usage = load('gauge_usage', variant)
    print(f"{variant:6s} models: mean singular-value ratio κ(G) {usage['shear_cond_w'].mean():.3f} | "
          f"mean variance of log rms(h_post) {usage['var_log_rms'].mean():.1e}")

## Figure 2: example gauges and the gauges of a free and a pinned model

In [ ]:
sys.path.insert(0, '..')                                  # repo root, relative to notebooks/
from matplotlib import cm
from src import processes as PR

TAG = "nonergodic_c2a06_4l_h4_s1"
PROC = "nonergodic_c2a06"
SKIP = 1                                   # drop the BOS position
RNG = np.random.default_rng(0)

# ---- ground truth by exact filtering of the shared contexts ------------------
proc = next(p for p in PR.all_sweep_processes() if p["name"] == PROC)
M, pi, eta = proc["M"], proc["pi"], proc["eta"]
edges = np.cumsum([0] + [c.shape[1] for c in proc["components"]])
seqs = np.load(os.path.join(RESULTS_DIR, 'contexts', f'{PROC}.npy'))
B, L = seqs.shape

logp = np.zeros((B, L + 1, M.shape[0]))
wcomp = np.zeros((B, L + 1, len(edges) - 1))
for b_i in range(B):
    b = pi.copy()
    for t in range(L + 1):
        p = np.einsum("i,zij,j->z", b, M, eta)
        p = np.clip(p / p.sum(), 1e-12, None)
        logp[b_i, t] = np.log(p)
        wcomp[b_i, t] = np.add.reduceat(b, edges[:-1])
        if t < L:
            b = b @ M[seqs[b_i, t]]
            b = b / max(b @ eta, 1e-300)

lp = logp[:, SKIP:].reshape(-1, 3)
clp = lp - lp.mean(-1, keepdims=True)                      # centered log-NTP
VIR3 = cm.viridis(np.array([0.12, 0.52, 0.88]))[:, :3]     # the 3-color palette
ntp = np.exp(lp)                                           # ground-truth NTP over the 3 symbols
ntp = ntp / ntp.sum(-1, keepdims=True)
COL = np.clip(ntp @ VIR3, 0, 1)                            # color = NTP blended over the palette
N = len(clp)
tpos = np.tile(np.arange(SKIP, L + 1)[None, :], (B, 1)).reshape(-1)

U2 = np.stack([np.array([1, -1, 0]) / np.sqrt(2),
               np.array([1, 1, -2]) / np.sqrt(6)], axis=1)
gt2 = clp @ U2                                             # native 2D content

# ---- the trained models: a free and a pinned model with four layers and four heads ----
def load_model(grid):
    f = os.path.join(RESULTS_DIR, 'gauge_geometry', grid, 'gauge_geometry_v4_4l_h4.npz')
    d = np.load(f, allow_pickle=True)
    i = [t for t in d["tags"]].index(TAG)
    
    def native2(key):
        return d[key][i][:, SKIP:].reshape(-1, 2).astype(np.float64)
    return dict(
        logits=d["logits"][i][:, SKIP:, :3].reshape(-1, 3).astype(np.float64),
        fin2=native2("coords_fin2"),
        post2=native2("coords_post2"),
    )

emp = load_model("free")
emp_p = load_model("pinned")

# ---- example gauges ----------------------------------------------------------
def rot(a):
    a = np.deg2rad(a)
    return np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])

SHEAR30 = np.array([[1.0, np.tan(np.deg2rad(30))], [0.0, 1.0]])
ROTSTRETCH = rot(45) @ np.diag([1.5, 0.6])
LAM_POS = np.where(tpos < 0.5 * (L + 1), 0.35, 1.9)   # first vs second half of the context
LAM_RND = RNG.uniform(0.0, 6.0, N)

ROWS = [
    ("logits", "3d", [
        ("Centered log-NTP", clp),
        ("log-NTP", lp),
        ("Random shift\nper position", clp + RNG.uniform(-1.2, 1.2, (N, 1)) * np.ones(3)),
        ("Model (free)", emp["logits"]),
        ("Model (pinned)", emp_p["logits"]),
    ]),
    ("resid_final", "2d", [
        ("No linear map", gt2),
        ("Shear (30\u00b0)", gt2 @ SHEAR30.T),
        ("Rotation (45\u00b0)\nstretch (1.5, 0.6)", gt2 @ ROTSTRETCH.T),
        ("Model (free)", emp["fin2"]),
        ("Model (pinned)", emp_p["fin2"]),
    ]),
    ("resid_post", "2d", [
        ("No per-position\nscaling", gt2),
        ("Random scaling\nper position", gt2 * LAM_RND[:, None]),
        ("Scaling based\non position", gt2 * LAM_POS[:, None]),
        ("Model (free)", emp["post2"]),
        ("Model (pinned)", emp_p["post2"]),
    ]),
]

W, H = 5.5, 3.95
fig = plt.figure(figsize=(W, H))
gs = fig.add_gridspec(3, 6, width_ratios=[1, 1, 1, 0.16, 1, 1],
                      left=0.062, right=0.995, top=0.885, bottom=0.015,
                      hspace=0.26, wspace=0.10)

fig.text(0.062 + 0.5 * (0.60 - 0.062), 0.912, "Ground-Truth (example gauges)",
         ha="center", va="center", fontsize=8.5)
fig.text(0.815, 0.912, "Empirical", ha="center", va="center", fontsize=8.5)
_b, _t, _l, _r = gs.get_grid_positions(fig)
_xsep = 0.5 * (_l[3] + _r[3])
fig.add_artist(plt.Line2D([_xsep, _xsep], [0.02, 0.888], color="0.7", lw=0.7,
                          transform=fig.transFigure))

for r, (hook, kind, panels) in enumerate(ROWS):
    for c, (title, pts) in enumerate(panels):
        col = c if c < 3 else c + 1
        if kind == "3d" :
            ax = fig.add_subplot(gs[r, col], projection="3d")
            ax.set_facecolor("none")
            ax.patch.set_alpha(0.0)
            f = 1.20
            bb = ax.get_position()
            ax.set_position([bb.x0 - 0.5 * (f - 1) * bb.width,
                             bb.y0 - 0.5 * (f - 1) * bb.height - 0.030,
                             f * bb.width, f * bb.height])
            ax.view_init(elev=6, azim=-35)
            if pts is not None:
                ctr = pts.mean(0)
                half = 0.55 * (pts.max(0) - pts.min(0)).max()
                lo3, hi3 = ctr - half, ctr + half
                ax.plot([lo3[0], hi3[0]], [lo3[1], hi3[1]], [lo3[2], hi3[2]],
                        color="0.75", lw=0.6, zorder=0)
                ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=COL, s=2.0,
                           linewidths=0, alpha=0.85)
                ax.set_xlim(lo3[0], hi3[0])
                ax.set_ylim(lo3[1], hi3[1])
                ax.set_zlim(lo3[2], hi3[2])
            ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
            for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
                axis.pane.set_edgecolor("0.85")
                axis.pane.set_alpha(0.12)
                axis.line.set_color("0.65")
                axis.line.set_linewidth(0.6)
            ax.set_box_aspect((1, 1, 1))
        else:
            ax = fig.add_subplot(gs[r, col])
            if pts is not None:
                ax.scatter(pts[:, 0], pts[:, 1], c=COL, s=2.4, linewidths=0,
                           alpha=0.85)
                m = 0.06 * (pts.max(0) - pts.min(0)).max()
                ax.set_xlim(pts[:, 0].min() - m, pts[:, 0].max() + m)
                ax.set_ylim(pts[:, 1].min() - m, pts[:, 1].max() + m)
            ax.set_xticks([]); ax.set_yticks([])
            for s_ in ax.spines.values():
                s_.set_linewidth(0.5); s_.set_color("0.6")
            ax.set_box_aspect(1)
        
        ax.set_title(title, fontsize=6.2, pad=2, y=0.90 if kind == "3d" else None)
        if c == 0:
            fig.text(0.026, {0: 0.775, 1: 0.46, 2: 0.155}[r], hook,
                     rotation=90, ha="center", va="center", fontsize=9,
                     family="monospace")

fig.savefig(f'{PLOT_DIR}/figure2.pdf')
plt.show()
plt.close()

## Appendix H: comparison with Boundless DAS

In [ ]:
bdas_free, bdas_pinned, bdas_all = load('boundless_das', 'free'), load('boundless_das', 'pinned'), load_all('boundless_das')
print("resid_final, all models")
print(f"    median cosine of the largest principal angle to the derived subspace: {np.median(bdas_all['final_content_svmin']):.2f}")
print(f"    median number of additional directions: {np.median(bdas_all['final_n_extra']):g}")
print(f"    largest KL from interchanging only the additional directions: {np.abs(bdas_all['final_extra_kl_raw']).max():.1e} nats")
print("resid_post, free models")
print(f"    median cosine of the largest principal angle to the derived subspace: {np.median(bdas_free['post_content_svmin']):.2f}")
print(f"    median number of additional directions: {np.median(bdas_free['post_n_extra']):g}")
print(f"    median KL from interchanging only the additional directions: {np.median(bdas_free['post_extra_kl']):.1e} nats")
print(f"    the same with the length of the orthogonal component restored: {np.median(bdas_free['post_extra_kl_normfix']):.1e} nats")
print("resid_post, pinned models")
print(f"    median KL from interchanging only the additional directions: {np.median(bdas_pinned['post_extra_kl']):.1e} nats")

## Appendix H: the orthogonal component of resid_post at three times its length

In [ ]:
rescale = load('length_rescaling', 'free')
print(f"free models, orthogonal component rescaled by {float(np.load(sorted(glob.glob(os.path.join(RESULTS_DIR, 'length_rescaling', 'free', '*.npz')))[0])['scale']):g}: "
      f"median KL {np.median(rescale['resid_post_comp_scale']):.1e} nats")